# Toolrecover: DPO and Bradley–Terry preference experiments
Run on a Colab GPU (Runtime → Change runtime type → T4). This notebook has **not yet been GPU-executed**. It uses actual pretrained causal LMs, full-parameter DPO with a frozen reference, and a separate scalar reward model. It does not implement PPO RLHF.

Public human preference labels come from [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) (MIT; retain upstream notices). These are general helpfulness/harmlessness comparisons, not tool recovery tasks. Model-generated rejudgments are separately marked AI; no person is claimed to have labeled this experiment. The local integration report uses only scripted synthetic labels.

Each training CLI invocation is a separate process, releasing model and GPU memory on completion. Use validation for hyperparameters; do not tune to the reported test. Tiny subsets and single seeds cannot support broad improvement claims.

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/content/toolrecover')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/hiroshi-os/toolrecover.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[preferences]'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.cuda.get_device_name(0))
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True))
def run(*args):
    subprocess.run([sys.executable, '-m', 'toolrecover.preferences', *map(str, args)], check=True)


## Configure and obtain labels
The default streams 100 unique prompts from the original HH train split; it does not download the whole dataset. A deterministic grouped 80/10/10 split keeps normalized identical prompts together. This is our subset split, not HH's published test. Alternatively set `USE_PUBLIC=False` and upload your own schema-valid JSONL; its human/AI/scripted provenance and source/license must be explicit.

Overlength rows are explicitly excluded, counted and written to an audit before training; remaining splits must all be nonempty. This filtering changes the measured population.

In [ ]:
MODEL = 'HuggingFaceTB/SmolLM2-135M-Instruct'
JUDGE = 'Qwen/Qwen2.5-0.5B-Instruct'
COUNT, STEPS, SEED, MAX_LENGTH = 100, 20, 0, 512
USE_PUBLIC = True
if USE_PUBLIC:
    run('import-hh', '--count', COUNT, '--seed', SEED, '--out', 'artifacts/preferences/raw.jsonl')
else:
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload one JSONL file'
    run('import-jsonl', '--data', next(iter(uploaded)), '--out', 'artifacts/preferences/raw.jsonl')
from transformers import AutoTokenizer
from toolrecover.preferences import read_rows, write_rows, batch_pairs, assign_splits
import json
raw = read_rows('artifacts/preferences/raw.jsonl')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
kept, excluded = [], []
for row in raw:
    try:
        batch_pairs(tokenizer, [row], MAX_LENGTH)
        kept.append(row)
    except ValueError as error:
        excluded.append({'id': row['id'], 'reason': str(error)})
Path('artifacts/preferences/filter-audit.json').write_text(json.dumps(excluded, indent=2))
assert len({r['prompt'] for r in kept}) >= 3, 'Increase COUNT or MAX_LENGTH'
kept = assign_splits(kept, SEED)  # assign final splits after explicit length filtering
write_rows('artifacts/preferences/human-or-user.jsonl', kept)
print({'kept': len(kept), 'excluded': len(excluded)})
run('validate', '--data', 'artifacts/preferences/human-or-user.jsonl')


## Train DPO and the separate reward model
DPO sums log probabilities over completion tokens including EOS, excluding prompts and padding. Tokenization explicitly concatenates separately tokenized prompt/completion segments. Its stable loss is `-logsigmoid(beta*((logπchosen-logπrejected)-(logπref_chosen-logπref_rejected)))`. Reference weights are frozen. The reward model minimizes `-logsigmoid(rchosen-rrejected)`. Both use fixed final checkpoints, held-out ranking, loss and label counts. DPO pair_accuracy reports raw completion-log-probability ranking; dpo_improvement_rate separately measures reference-relative margin improvement. Neither measures generated-answer quality or calibrated reward accuracy. Summed log probabilities can favor shorter responses.

In [ ]:
DATA = 'artifacts/preferences/human-or-user.jsonl'
for method in ['dpo', 'reward']:
    run('train', '--data', DATA, '--out', f'artifacts/preferences/{method}',
        '--method', method, '--model-name', MODEL, '--steps', STEPS, '--seed', SEED,
        '--batch-size', 1, '--max-length', MAX_LENGTH, '--device', 'cuda')
    run('eval', '--data', DATA, '--checkpoint', f'artifacts/preferences/{method}', '--device', 'cuda')


## Optional actual AI preference collection and DPO
This uses local Hugging Face generation, with no paid API. The judge sees only the prompt and randomized A/B candidate texts, without original winner labels. The audit records permutations, raw output and rejected judgments. Only exact JSON A/B outputs are accepted; ties/prose/invalid output do not become training labels. An uncalibrated small judge can be wrong and exhibit position bias. Randomized order does not remove that bias. AI DPO is pending unless these cells execute successfully. Human source labels remain under candidate provenance and are not used as AI labels.

Rejudging held-out rows allows measurement of agreement with that judge only. It does not independently establish human preference improvement.

In [ ]:
RUN_AI = False  # Set True to actually collect AI judgments and train on accepted pairs.
if RUN_AI:
    AI_DATA = 'artifacts/preferences/ai.jsonl'
    run('judge', '--data', DATA, '--out', AI_DATA, '--model-name', JUDGE, '--device', 'cuda', '--seed', SEED)
    accepted = read_rows(AI_DATA)
    assert {r['split'] for r in accepted} == {'train', 'validation', 'test'}, 'Judge rejected an entire split; inspect audit'
    run('train', '--data', AI_DATA, '--out', 'artifacts/preferences/ai-dpo', '--method', 'dpo',
        '--model-name', MODEL, '--steps', STEPS, '--batch-size', 1, '--max-length', MAX_LENGTH, '--device', 'cuda')


## Download results and checkpoints
Reports include source label counts and dataset hashes. Retain input JSONL and audits with the models. Checkpoints include the complete frozen DPO reference and can be large. Base/model/dataset licenses apply separately from this repository's MIT code license.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/toolrecover-preferences', 'zip', 'artifacts/preferences')
files.download(archive)
